Sel 1: muat data

In [ ]:
from pathlib import Path

from review_nlp.ingestion.transform import refilter
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from review_nlp.sentiment.labels import rating_to_label

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
FILE = ROOT / "data/raw/com.whatsapp_2026-08-19_2026-11-17.parquet"
df = pd.read_parquet(FILE)
df = refilter(df)
df["reviewed_at"] = pd.to_datetime(df["reviewed_at"])
print(df.shape)
df.dtypes

Sel 2: kualitas data dan alasan eksklusi

In [ ]:
status = df["exclude_reason"].fillna("dipakai")
print(status.value_counts())
print((status.value_counts(normalize=True) * 100).round(1))

used = df[df["exclude_reason"].isna()].copy()
print("\nDipakai:", len(used), "| teks duplikat:", used["text_raw"].duplicated().sum())

Sel 3: distribusi rating dan label

In [ ]:
counts = used["rating"].value_counts().sort_index()
ax = counts.plot(kind="bar", title="Distribusi rating (ulasan dipakai)")
ax.set_xlabel("Bintang")
ax.set_ylabel("Jumlah")
plt.savefig(ROOT / "docs/images/rating_dist.png", dpi=150, bbox_inches="tight")
plt.show()

print((counts / counts.sum()).round(3))
used["label"] = used["rating"].map(rating_to_label)
print("\nProporsi label:")
print(used["label"].value_counts(normalize=True).round(3))

Sel 4: ulasan pendek vs ulasan dipakai (bias seleksi)

In [ ]:
short = df[df["exclude_reason"] == "too_short"]
perbandingan = pd.DataFrame(
    {
        "dipakai": used["rating"].value_counts(normalize=True).sort_index(),
        "too_short": short["rating"].value_counts(normalize=True).sort_index(),
    }
).round(3)
perbandingan

Sel 5: tren per minggu (proporsi sentimen)

In [ ]:
weekly = used.groupby("week_index").agg(n=("rating", "size"), rating_mean=("rating", "mean"))
share = used.groupby("week_index")["label"].value_counts(normalize=True).unstack().fillna(0)

margin = (1.96 * np.sqrt(share["negatif"] * (1 - share["negatif"]) / weekly["n"]) * 100).round(1)
print("Ulasan dipakai per minggu: min", weekly["n"].min(), "| maks", weekly["n"].max())
print("Margin kira-kira ±95% proporsi negatif (poin persen): min", margin.min(), "| maks", margin.max())

for d in (weekly, share):
    d.index = pd.Timestamp("2026-08-19") + pd.to_timedelta(d.index * 7, unit="D")

fig, ax = plt.subplots(1, 2, figsize=(12, 3.5))
weekly["rating_mean"].plot(ax=ax[0], marker="o", title="Rata-rata rating per minggu")
share.plot(ax=ax[1], marker="o", title="Proporsi sentimen (label bintang) per minggu")
fig.savefig(ROOT / "docs/images/weekly_trend.png", dpi=150, bbox_inches="tight")
plt.show()
share.round(3)

Sel 6: volume asli per minggu dan cakupan hari

In [ ]:
VOL = ROOT / "data/raw/com.whatsapp_2026-08-19_2026-11-17_weekly_volume.csv"
vol = pd.read_csv(VOL)
vol["minggu_mulai"] = pd.Timestamp("2026-08-19") + pd.to_timedelta(vol["week_index"] * 7, unit="D")
ax = vol.plot(
    x="minggu_mulai",
    y="ulasan_tersedia",
    marker="o",
    legend=False,
    title="Volume ulasan tersedia per minggu (sebelum kuota)",
)
plt.savefig(ROOT / "docs/images/weekly_volume.png", dpi=150, bbox_inches="tight")
plt.show()

per_day = df.groupby(df["reviewed_at"].dt.date).size()
print("Hari terisi:", len(per_day), "dari 91 | per hari: min", per_day.min(), "| maks", per_day.max())
vol

Sel 7: panjang teks

In [ ]:
used["n_words"] = used["text_raw"].str.split().str.len()
print(used["n_words"].describe().round(1))
ax = used["n_words"].clip(upper=100).plot(
    kind="hist", bins=40, title="Panjang ulasan (kata, dipotong di 100)"
)
plt.show()

Sel 8: sebaran versi aplikasi

In [ ]:
vc = used["app_version"].value_counts()
print("Tanpa versi:", round(used["app_version"].isna().mean(), 3))
print("Jumlah versi:", len(vc), "| versi dengan >= 200 ulasan:", int((vc >= 200).sum()))

by_version = (
    used.groupby("app_version")
    .agg(n=("rating", "size"), rating_mean=("rating", "mean"))
    .query("n >= 200")
    .sort_values("n", ascending=False)
)
by_version.round(2)

Sel 9: periksa filter bahasa (baca 50 contoh)

In [ ]:
dropped = df[df["exclude_reason"] == "non_indonesian"]
print("Dibuang karena bukan bahasa Indonesia:", len(dropped))
dropped[["lang", "text_raw"]].sample(min(50, len(dropped)), random_state=42)

Sel 10: hari tersibuk

In [ ]:
harian = (
    used.groupby([used["week_index"], used["reviewed_at"].dt.date])
    .agg(n=("rating", "size"), negatif=("label", lambda s: (s == "negatif").mean()))
    .reset_index()
)
harian["porsi_minggu"] = harian["n"] / harian.groupby("week_index")["n"].transform("sum")
harian.sort_values("n", ascending=False).head(8).round(3)

In [ ]:
hari = harian.sort_values("n", ascending=False).iloc[0]["reviewed_at"]
sub = used[(used["reviewed_at"].dt.date == hari) & (used["label"] == "negatif")]
print("Hari tersibuk:", hari, "| ulasan negatif:", len(sub))
sub[["rating", "text_raw"]].sample(min(20, len(sub)), random_state=0)

Sel 11: apakah ulasan tanpa versi bias

In [ ]:
used["tanpa_versi"] = used["app_version"].isna()
print("Proporsi tanpa versi menurut bintang:")
print(used.groupby("rating")["tanpa_versi"].mean().round(3))
print("\nProporsi tanpa versi menurut minggu:")
print(used.groupby("week_index")["tanpa_versi"].mean().round(3).to_string())

Sel 12: tabel 11 versi yang layak dibandingkan

In [ ]:
tabel = (
    used.dropna(subset=["app_version"])
    .groupby("app_version")
    .agg(
        n=("rating", "size"),
        awal=("reviewed_at", "min"),
        median=("reviewed_at", "median"),
        akhir=("reviewed_at", "max"),
        rating_mean=("rating", "mean"),
        negatif=("label", lambda s: (s == "negatif").mean()),
    )
    .query("n >= 200")
    .sort_values("median")
)
tabel.round(3)

In [ ]:
def bandingkan(a, b):
    xa = used.loc[used["app_version"] == a, "label"] == "negatif"
    xb = used.loc[used["app_version"] == b, "label"] == "negatif"
    pa, pb, na, nb = xa.mean(), xb.mean(), len(xa), len(xb)
    se = np.sqrt(pa * (1 - pa) / na + pb * (1 - pb) / nb)
    d = pb - pa
    return {
        "versi_a": a, "versi_b": b, "n_a": na, "n_b": nb,
        "neg_a": round(pa, 3), "neg_b": round(pb, 3),
        "selisih": round(d, 3), "ci95": (round(d - 1.96 * se, 3), round(d + 1.96 * se, 3)),
    }
    

Sel 13: diagnosis 9 September

In [ ]:
H = pd.Timestamp("2026-09-09").date()
hari = df[df["reviewed_at"].dt.date == H]
lain = df[(df["week_index"] == 3) & (df["reviewed_at"].dt.date != H)]
print("Ulasan di sampel: 9 Sep =", len(hari), "| sisa minggu 3 =", len(lain))
print(
    pd.DataFrame(
        {
            "9_sep": hari["rating"].value_counts(normalize=True).sort_index(),
            "sisa_minggu_3": lain["rating"].value_counts(normalize=True).sort_index(),
        }
    ).round(3)
)
print("too_short:", round((hari["exclude_reason"] == "too_short").mean(), 3),
      "vs", round((lain["exclude_reason"] == "too_short").mean(), 3))
print("Teks duplikat pada 9 Sep:", int(hari["text_raw"].duplicated().sum()))
print(hari["app_version"].value_counts(dropna=False).head(5))
hari[["rating", "text_raw"]].sample(20, random_state=0)

Sel 14: proporsi berbobot volume

In [ ]:
vol = pd.read_csv(VOL)
bobot = vol.set_index("week_index")["ulasan_tersedia"] / vol.set_index("week_index")["ulasan_diambil"]
used["bobot"] = used["week_index"].map(bobot)
berbobot = used.groupby("label")["bobot"].sum() / used["bobot"].sum()
pd.DataFrame(
    {"tanpa_bobot": used["label"].value_counts(normalize=True), "berbobot_volume": berbobot}
).round(3)

Sel 15: negatif per versi, dengan kontrol waktu

In [ ]:
v = used[used["app_version"].isin(tabel.index)]
per_versi = (
    v.groupby("app_version")["label"]
    .agg(n="size", neg=lambda s: (s == "negatif").mean())
    .reindex(tabel.index)
)
per_versi["moe"] = 1.96 * np.sqrt(per_versi["neg"] * (1 - per_versi["neg"]) / per_versi["n"])

fig, ax = plt.subplots(figsize=(9, 3.5))
ax.errorbar(range(len(per_versi)), per_versi["neg"], yerr=per_versi["moe"], fmt="o-", capsize=3)
ax.set_xticks(range(len(per_versi)))
ax.set_xticklabels(per_versi.index, rotation=45, ha="right")
ax.set_title("Proporsi negatif per versi (label bintang, selang 95%)")
fig.savefig(ROOT / "docs/images/negatif_per_versi.png", dpi=150, bbox_inches="tight")
plt.show()

# kontrol waktu: hanya ulasan 3 minggu terakhir, versi lama vs baru
akhir = used[(used["week_index"] >= 10) & used["app_version"].isin(tabel.index)].copy()
akhir["kelompok"] = np.where(akhir["app_version"].isin(tabel.index[:5]), "lama", "baru")
print(akhir.groupby("kelompok")["label"].agg(n="size", neg=lambda s: (s == "negatif").mean()).round(3))

Sel 16: karakterisasi lonjakan 9 September (akhir notebook 01)

In [ ]:
V = "2.26.27.85"
sv = df[df["app_version"] == V]
print("Ulasan versi", V, "di sampel:", len(sv), "| pada 9 Sep:", int((sv["reviewed_at"].dt.date == H).sum()))
print(sv.groupby(sv["reviewed_at"].dt.date).size().sort_values(ascending=False).head(5))

burst = sv[sv["reviewed_at"].dt.date == H]
print("\nSebaran jam (9 Sep):")
print(burst.groupby(burst["reviewed_at"].dt.hour).size().to_string())
print("\nNama pengguna unik:", burst["user_hash"].nunique(), "dari", len(burst))
print("Teks unik:", burst["text_raw"].nunique(), "dari", len(burst))
print("Bintang:", burst["rating"].value_counts(normalize=True).sort_index().round(3).to_dict())
print(burst["text_raw"].value_counts().head(8))

per_hari = df.groupby(df["reviewed_at"].dt.date)["text_raw"].apply(lambda s: s.duplicated().mean())
print("\nRasio teks duplikat per hari:")
print(per_hari.describe().round(3))
print("9 Sep:", round(per_hari[H], 3))

### Pembaruan: filter bahasa dan 9 September
- Setelah perbaikan (deteksi huruf kecil + kata penanda Indonesia), ulasan non_indonesian turun dari
  158 menjadi 80. Dari 50 contoh baru, 5 (10%) sebenarnya Indonesia/campuran; dampak ke data < 0,2%.
- 9 Sep: 137 dari 358 ulasan berasal dari versi 2.26.27.85 (307 ulasan total versi itu), tersebar
  sepanjang hari (maks 16 per jam). Rasio teks duplikat tertinggi dari semua hari (0,243 vs kuartil
  ketiga 0,148). Penyebab tidak dapat dipastikan. Menghapus hari itu dari data latih tidak
  mengubah macro-F1 (selisih 0,005).
- Proporsi negatif turun dari ~0,50 (minggu 0-1) ke ~0,33 (minggu 11-12). Efek versi tidak dapat
  dipisahkan dari efek waktu (versi lama hanya 21 ulasan di 3 minggu terakhir).